In [104]:
# 2. 소득 예측 모델 만들기

In [105]:
import pandas as pd

In [106]:
# adult.csv 데이터는 미국인긔 성별, 인종, 학력 등 다양한 인적 정보를 담고 있는 인구 조사 데이터
# 해당 데이터를 이용해 인적 정보로 소득을 예측하는 의사나무결정 모델을 만듬

In [107]:
# 모델을 만드는 절차
# 전처리 -> 모델 만들기 -> 예측 및 성능 평가

In [108]:
df = pd.read_csv('input/adult.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             48842 non-null  int64
 1   workclass       48842 non-null  str  
 2   fnlwgt          48842 non-null  int64
 3   education       48842 non-null  str  
 4   education_num   48842 non-null  int64
 5   marital_status  48842 non-null  str  
 6   occupation      48842 non-null  str  
 7   relationship    48842 non-null  str  
 8   race            48842 non-null  str  
 9   sex             48842 non-null  str  
 10  capital_gain    48842 non-null  int64
 11  capital_loss    48842 non-null  int64
 12  hours_per_week  48842 non-null  int64
 13  native_country  48842 non-null  str  
 14  income          48842 non-null  str  
dtypes: int64(6), str(9)
memory usage: 5.6 MB


In [109]:
# adult 데이터는 48,842명의 정보를 담고 있으면 변수가 15개
# 이 중 연소득을 나타낸 income을 타겟 변수, 나머지 14개를 예측 변수로 사용

In [110]:
# * 소득 예측 모델을 어디에 활용할까?
# 1) 고가 상품 구매 프로모션
# 소득 예측 모델은 고급 수입차나 명품 시계처럼 고가의 상품을 구매하도록 독려하는 프로모션에 활용할 수 있음
# 소득이 높아 상품을 구매할 가능성이 높은 사람을 예측하면 프로모션 활동에 드는 시간과 비용을 절약할 수 있음

In [111]:
# 2) 저소득층 지원 대상 확대
# 정부에서 저소득층 지원 사업 대상자를 찾아낼 때에도 소득 예측 모델을 활용할 수 있음
# 저소득층 선제적으로 찾아 지원하면 사람들이 스스로 지원 요건을 확인하고 신청하게 할 때보다
# 복지 프로그램 대상을 확대할 수 있음

In [112]:
# 01. 전처리하기
# 1) 타겟 변수 전처리
# 타겟 변수 income을 검토하고 전처리
# income은 조사 응답자의 연소득이 5만 달러를 초과하는지 여부를 나타냄

In [113]:
print(df['income'].value_counts(normalize=True))
# ** normalize=True 로 두는 이유는 1을 기준으로 비율로서 확인히 가능하기 때문
# 중요

income
<=50K    0.760718
>50K     0.239282
Name: proportion, dtype: float64


In [114]:
# 연소득이 5만 달러를 초과하는 사람 (>50k)는 23.9%, 5만 달러 이하인 사람 (<=50k)은 76%
# 변수의 값에 특수 문자나 대소문자가 섞여 있으면 다루기 불편하므로 5만 달러 초과하면 'high',
# 그렇지 않으면 'low'로 값을 수정

In [115]:
import numpy as np

df['income'] = np.where(df['income'] == '>50K', 'high', 'low')
print(df['income'].value_counts(normalize=True))

income
low     0.760718
high    0.239282
Name: proportion, dtype: float64


In [116]:
# 2. 불필요한 변수 제거하기
# 이름, 아이디, 주소 같은 변수는 대부분 값이 고유값이어서 반복되는 패턴이 없고 타겟 변수와도 관련성이 없음
# 이런 변수는 타겟 변수를 예측하는 데 도움이 되지 않고 모델링 시간만 늘이는 역할을 하므로 제거

# fnlwgt는 adult 데이터를 이용해 미국의 실제 연구를 추정할 대 사용하는 가중치
# 타겟 변수를 예측하는데 도움이 되지 않기 때문에 제거
df = df.drop(columns=['fnlwgt'])

In [117]:
# 3. 문자 타입 변수를 숫자 타입으로 바꾸기
# 모델을 만드는데 사용되는 모든 변수는 숫자 타입이어야 함
# df.info() 실행 결과 Dtype이 int64(6), str(9)만 문자타입 변수들이 있는데
# 이 변수들을 모델을 활용하는데 사용하려면 숫자 타입으로 바꿔야 함

In [118]:
# 1) 원 핫 인코딩하기
# 원핫 인코딩 one-hot encoding: 값을 1과 0으로 바꾸는 방법
# 변수의 범주가 특정 값이면 1, 그렇지 않으면 0으로 바꾸는 문자타입을 숫자타입으로 만들 수 있음

In [119]:
# 성별을 추출해 원핫 인코딩
df_tmp = df[['sex']]
df_tmp.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   sex     48842 non-null  str  
dtypes: str(1)
memory usage: 381.7 KB


In [120]:
# pd.get_dummies(0에 데이터 프레임을 입력하면 문자 타입 변수를 원핫 인코딩을 적용해 변환
# df_tmp의 문자 타입 변수에 원핫 인코딩 적용
df_tmp = pd.get_dummies(df_tmp)
df_tmp.info()


<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   sex_Female  48842 non-null  bool 
 1   sex_Male    48842 non-null  bool 
dtypes: bool(2)
memory usage: 95.5 KB


In [121]:
# 출력 결과를 보면 sex가 사라지고 그 대신 sex_Female, sex_Male이 만들어 짐
# 원핫 인코딩으로 만들어진 변수의 데이터 타입은 bool
# bool은 False나 True냐를 나누는 데이터 타입

# sex_Female은 sex가 Female이면 1, 그렇지 않으면 0으로 된 변수
# sex_Male은 반대로 sex가 Male이면 1, 그렇지 않으면 0으로 된 변수
# 두 변수 중 한쪽이 1이면 다른 한 쪽은 반드시 0

print(df_tmp[['sex_Female', 'sex_Male']].head())

   sex_Female  sex_Male
0       False      True
1       False      True
2       False      True
3       False      True
4        True     False


In [122]:
# df에 원핫 인코딩 적용
# 타겟 변수인 income만 원래대로 유지하고, 모든 문자 타입 변수를 원핫 인코딩 적용
target = df['income']
print(target)

0         low
1         low
2        high
3        high
4         low
         ... 
48837     low
48838    high
48839     low
48840     low
48841    high
Name: income, Length: 48842, dtype: str


In [123]:
df = df.drop(columns=['income'])
df = pd.get_dummies(df)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Columns: 107 entries, age to native_country_Yugoslavia
dtypes: bool(102), int64(5)
memory usage: 6.6 MB


In [130]:
# df.info() 출력 결과에 개별 변수의 정보가 출력되지 않은 이유는
# 변수가 100개 이하일 때만 변수 정보를 출력하도록 설정되어 있기 때문
# df.info()에 max_cols=npl.inf를 입력하면 변수의 수와 관계없이 모든 변수의 정보를 출력
print(df.info(max_cols=np.inf))

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 107 columns):
 #    Column                                     Non-Null Count  Dtype
---   ------                                     --------------  -----
 0    age                                        48842 non-null  int64
 1    education_num                              48842 non-null  int64
 2    capital_gain                               48842 non-null  int64
 3    capital_loss                               48842 non-null  int64
 4    hours_per_week                             48842 non-null  int64
 5    workclass_?                                48842 non-null  bool 
 6    workclass_Federal-gov                      48842 non-null  bool 
 7    workclass_Local-gov                        48842 non-null  bool 
 8    workclass_Never-worked                     48842 non-null  bool 
 9    workclass_Private                          48842 non-null  bool 
 10   workclass_Self-emp-inc                     

In [131]:
# 출력 결과를 보면 변수가 108?개로 늘어나고, 문자 타입 변수가 전부 숫자 타입으로 변한것을 확인
df.to_csv('./data/df_step_01.csv')

OSError: Cannot save file into a non-existent directory: 'data'